This notebook loads translation quality data and detector inference outputs, aligns them, and analyzes how human/MT quality metrics correlate with MT-detection labels.

- config paths, languages, detector/metric mappings
- helper functions: filename parsing, label normalization, correlation, binning, LaTeX export
- load quality CSVs into long format (`collect_quality_files`)
- load inference CSVs into long format (`collect_inference_files`)
- merge quality + predictions (`merge_quality_and_predictions`)
- compute correlation stats per language/dataset/detector for `bleurt`/`comet`
- bin quality scores into Low/Medium/High and compute detection rates
- per-translator quality + detection analysis
- generate bar/scatter plots and summary CSV/LaTeX outputs in `quality_analysis_outputs`

In [ ]:
import os
import re
from pathlib import Path
from typing import Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr


BASE_DIR = Path("../")

QUALITY_ROOT = BASE_DIR / "Quality scores"
INFERENCE_ROOT = BASE_DIR / "Evaluation final/Inference"
OUTPUT_ROOT = BASE_DIR / "quality_analysis_outputs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LANGS = ["en", "es", "af", "sw"]
TRANSLATORS = ["claude", "gemini", "gemma", "llamma", "nllb", "openai", "qwen"]
HUMAN_COL = "human translation"

LANGUAGE_NAMES = {
    "en": "English",
    "es": "Spanish",
    "af": "Afrikaans",
    "sw": "Swahili",
}

DATASET_DOMAIN = {
    "eu_dgt": "legislative",
    "openwho": "medical",
    "smol": "conversational",
    "flores": "benchmark",
}

# detector folder -> paper-friendly detector name
DETECTOR_NAME_MAP = {
    "claude": "claude",
    "gemini": "gemini",
    "gpt5": "gpt5",
    "claude_few_shot": "few_shot_claude",
    "gemini_few_shot": "few_shot_gemini",
    "gpt5_few_shot": "few_shot_gpt5",
    "mdeberta": "mdeberta_sentence",
    "mdeberta chunk": "mdeberta_chunk300",
    "xlmr": "xlmr_sentence",
    "xlmr chunk": "xlmr_chunk300",
}

QUALITY_METRICS = ["bleu", "bleurt", "comet"]

In [ ]:
def sanitize_name(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]+", "_", text.strip().lower())

def normalize_label(x) -> Optional[int]:
    """
    Assumes:
      1 = MT
      0 = HT
    Change here if your labels are reversed.
    """
    if pd.isna(x):
        return None

    s = str(x).strip().lower()
    if s in {"1", "mt", "machine", "machine translated", "machine-translated"}:
        return 1
    if s in {"0", "ht", "human", "human translation", "human-translated"}:
        return 0

    try:
        v = int(float(s))
        if v in (0, 1):
            return v
    except Exception:
        pass

    return None

def safe_corr(x: pd.Series, y: pd.Series) -> Tuple[float, float, float, float]:
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return np.nan, np.nan, np.nan, np.nan
    pr, pp = pearsonr(x, y)
    sr, sp = spearmanr(x, y)
    return pr, pp, sr, sp

def make_quantile_bins(series: pd.Series, q: int = 3) -> pd.Series:
    labels = ["Low", "Medium", "High"] if q == 3 else [f"Q{i+1}" for i in range(q)]
    return pd.qcut(series, q=q, labels=labels, duplicates="drop")

def export_latex_table(df: pd.DataFrame, out_path: Path, caption: str, label: str, float_format="%.3f"):
    latex = df.to_latex(
        index=False,
        escape=False,
        float_format=lambda x: float_format % x if isinstance(x, (float, int, np.floating)) else str(x),
    )
    latex = latex.replace("\\begin{tabular}", "\\begin{center}\n\\small\n\\begin{tabular}")
    latex = latex.replace("\\end{tabular}", "\\end{tabular}\n\\end{center}")
    latex = f"\\begin{{table}}[t]\n{latex}\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\end{{table}}\n"
    out_path.write_text(latex, encoding="utf-8")

def parse_quality_filename(filename: str) -> Optional[Dict]:
    """
    Matches:
      en_eu_dgt_with_scores.csv
      af_flores_with_scores.csv
      af_flores_output_with_scores_mdeb_chunk.csv
      en_openwho_output_with_scores_xlmr_chunk.csv
    """
    name = filename.replace(".csv", "")

    # chunked
    m = re.match(r"^(en|es|af|sw)_(.+?)_output_with_scores_(mdeb|xlmr)_chunk$", name)
    if m:
        lang, dataset, chunk_model = m.groups()
        return {
            "lang": lang,
            "dataset": dataset,
            "is_chunked": True,
            "chunk_model": chunk_model,
        }

    # non-chunked
    m = re.match(r"^(en|es|af|sw)_(.+?)_with_scores$", name)
    if m:
        lang, dataset = m.groups()
        return {
            "lang": lang,
            "dataset": dataset,
            "is_chunked": False,
            "chunk_model": None,
        }

    return None

def parse_inference_filename(filename: str) -> Optional[Dict]:
    """
    Matches:
      model_claude_col_claude.csv
      model_claude_col_human translation.csv
      model_mdeberta_col_qwen.csv
    """
    name = filename.replace(".csv", "")

    m = re.match(r"^(?:(.+?)_)?model_(.+?)_col_(.+)$", name)
    if not m:
        return None

    dataset_prefix, model_name, col_name = m.groups()
    return {
        "dataset_prefix": dataset_prefix,
        "model_name_in_file": model_name,
        "translator_col": col_name,
    }

def canonical_detector(folder_name: str) -> Optional[str]:
    key = folder_name.strip().lower()
    return DETECTOR_NAME_MAP.get(key)

def canonical_chunk_model(chunk_model: Optional[str]) -> Optional[str]:
    if chunk_model == "mdeb":
        return "mdeberta_chunk300"
    if chunk_model == "xlmr":
        return "xlmr_chunk300"
    return None

def should_use_quality_file_for_detector(quality_info: Dict, detector: str) -> bool:
    if quality_info["is_chunked"]:
        return detector == canonical_chunk_model(quality_info["chunk_model"])
    else:
        return detector in {
            "zero_shot_claude",
            "zero_shot_gemini",
            "zero_shot_gpt5",
            "few_shot_claude",
            "few_shot_gemini",
            "few_shot_gpt5",
            "mdeberta_sentence",
            "xlmr_sentence",
        }

def load_quality_long(quality_path: Path) -> pd.DataFrame:
    info = parse_quality_filename(quality_path.name)
    if info is None:
        raise ValueError(f"Unrecognized quality filename: {quality_path.name}")

    lang = info["lang"]
    dataset = info["dataset"]
    language = LANGUAGE_NAMES[lang]
    domain = DATASET_DOMAIN.get(dataset, "unknown")

    df = pd.read_csv(quality_path)
    df = df.reset_index(drop=False).rename(columns={"index": "row_id_raw"})
    df["row_id_raw"] = df["row_id_raw"].astype(int)

    rows = []
    for tr in TRANSLATORS:
        row_df = pd.DataFrame({
            "row_id_raw": df["row_id_raw"],
            "human_translation": df[HUMAN_COL].astype(str),
            "translator": tr,
            "mt_text": df[tr].astype(str) if tr in df.columns else np.nan,
            "source_text": df["source text"].astype(str) if "source text" in df.columns else np.nan,
            "source_language": df["source language"].astype(str) if "source language" in df.columns else np.nan,
            "language": language,
            "lang_code": lang,
            "dataset": dataset,
            "domain": domain,
            "quality_file": quality_path.name,
            "is_chunked_quality": info["is_chunked"],
            "chunk_model": info["chunk_model"],
        })

        for metric in QUALITY_METRICS:
            col = f"{tr}_{metric}"
            row_df[metric] = pd.to_numeric(df[col], errors="coerce") if col in df.columns else np.nan

        rows.append(row_df)

    return pd.concat(rows, ignore_index=True)

def load_inference_long(inference_csv: Path, lang: str, dataset: str, detector_folder: str) -> pd.DataFrame:
    detector = canonical_detector(detector_folder)
    if detector is None:
        raise ValueError(f"Unknown detector folder: {detector_folder}")

    parsed = parse_inference_filename(inference_csv.name)
    if parsed is None:
        raise ValueError(f"Unrecognized inference filename: {inference_csv.name}")

    translator_col = parsed["translator_col"]
    df = pd.read_csv(inference_csv)

    label_col = None
    for candidate in ["label", "final_model_predicted_label"]:
        if candidate in df.columns:
            label_col = candidate
            break

    if HUMAN_COL not in df.columns or label_col is None:
        raise ValueError(
            f"Missing required columns in {inference_csv}. "
            f"Found columns: {list(df.columns)}"
        )

    df = df.reset_index(drop=False).rename(columns={"index": "row_id_raw"})
    df["row_id_raw"] = df["row_id_raw"].astype(int)
    df["human_translation"] = df[HUMAN_COL].astype(str)
    df["pred_mt"] = df[label_col].apply(normalize_label)
    df["translator"] = translator_col
    df["detector"] = detector
    df["lang_code"] = lang
    df["dataset"] = dataset
    df["detector_folder"] = detector_folder
    df["inference_file"] = inference_csv.name

    keep_cols = [
        "row_id_raw",
        "human_translation",
        "translator",
        "pred_mt",
        "detector",
        "lang_code",
        "dataset",
        "detector_folder",
        "inference_file",
    ]

    if "ht confidence" in df.columns:
        keep_cols.append("ht confidence")
    if "mt confidence" in df.columns:
        keep_cols.append("mt confidence")

    return df[keep_cols].copy()

def collect_quality_files() -> pd.DataFrame:
    all_rows = []
    for lang in LANGS:
        lang_dir = QUALITY_ROOT / lang
        if not lang_dir.exists():
            continue
        for csv_path in lang_dir.glob("*.csv"):
            info = parse_quality_filename(csv_path.name)
            if info is None:
                print(f"[WARN] Skipping unmatched quality file: {csv_path}")
                continue
            qlong = load_quality_long(csv_path)
            all_rows.append(qlong)

    if not all_rows:
        raise RuntimeError("No quality files found.")
    return pd.concat(all_rows, ignore_index=True)

def collect_inference_files() -> pd.DataFrame:
    all_rows = []
    for lang in LANGS:
        lang_dir = INFERENCE_ROOT / lang
        if not lang_dir.exists():
            continue

        for dataset_dir in lang_dir.iterdir():
            if not dataset_dir.is_dir():
                continue
            dataset = dataset_dir.name

            for detector_dir in dataset_dir.iterdir():
                if not detector_dir.is_dir():
                    continue
                detector_folder = detector_dir.name
                if canonical_detector(detector_folder) is None:
                    print(f"[WARN] Skipping unknown detector dir: {detector_dir}")
                    continue

                for csv_path in detector_dir.glob("*.csv"):
                    parsed = parse_inference_filename(csv_path.name)
                    if parsed is None:
                        print(f"[WARN] Skipping unmatched inference file: {csv_path}")
                        continue
                    try:
                        ilong = load_inference_long(csv_path, lang, dataset, detector_folder)
                        all_rows.append(ilong)
                    except Exception as e:
                        print(f"[WARN] Failed loading {csv_path}: {e}")

    if not all_rows:
        raise RuntimeError("No inference files found.")
    return pd.concat(all_rows, ignore_index=True)

def merge_quality_and_predictions(quality_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge strategy:
    1. Match on lang_code, dataset, translator, human_translation, row_id_raw
    2. Fallback: if human_translation is enough, use that
    """
    merged_all = []

    for detector in pred_df["detector"].dropna().unique():
        pred_sub = pred_df[pred_df["detector"] == detector].copy()

        # only use compatible quality files
        qual_sub = quality_df[
            quality_df.apply(lambda r: should_use_quality_file_for_detector(
                {"is_chunked": r["is_chunked_quality"], "chunk_model": r["chunk_model"]},
                detector
            ), axis=1)
        ].copy()

        # exact merge
        merged = qual_sub.merge(
            pred_sub,
            on=["lang_code", "dataset", "translator", "human_translation", "row_id_raw"],
            how="inner",
            suffixes=("", "_pred")
        )

        # fallback if exact merge fails for some files
        if len(merged) == 0:
            merged = qual_sub.merge(
                pred_sub,
                on=["lang_code", "dataset", "translator", "human_translation"],
                how="inner",
                suffixes=("", "_pred")
            )

        if len(merged) > 0:
            merged["detector"] = detector
            merged_all.append(merged)

    if not merged_all:
        raise RuntimeError("No merged rows found between quality and predictions.")

    merged_df = pd.concat(merged_all, ignore_index=True)
    merged_df = merged_df[merged_df["pred_mt"].notna()].copy()
    merged_df["pred_mt"] = merged_df["pred_mt"].astype(int)
    return merged_df



In [ ]:
print("Loading quality files...")
quality_long = collect_quality_files()

print("Loading inference files...")
pred_long = collect_inference_files()

print("Merging...")
merged = merge_quality_and_predictions(quality_long, pred_long)

merged.to_csv(OUTPUT_ROOT / "merged_quality_predictions.csv", index=False)

print(f"Merged rows: {len(merged)}")

In [ ]:
corr_rows = []
for metric in ["bleurt", "comet"]:
    sub_metric = merged[merged[metric].notna()].copy()
    for (language, dataset, detector), sub in sub_metric.groupby(["language", "dataset", "detector"]):
        pr, pp, sr, sp = safe_corr(sub[metric], sub["pred_mt"])
        corr_rows.append({
            "language": language,
            "dataset": dataset,
            "detector": detector,
            "metric": metric.upper(),
            "n": len(sub),
            "pearson_r": pr,
            "pearson_p": pp,
            "spearman_r": sr,
            "spearman_p": sp,
        })

corr_df = pd.DataFrame(corr_rows).sort_values(["metric", "language", "dataset", "detector"])
corr_df.to_csv(OUTPUT_ROOT / "correlation_results.csv", index=False)

corr_paper = (
    corr_df.pivot_table(
        index=["language", "dataset", "detector"],
        columns="metric",
        values="spearman_r"
    )
    .reset_index()
)
corr_paper.to_csv(OUTPUT_ROOT / "correlation_results_paper.csv", index=False)

export_latex_table(
    corr_paper,
    OUTPUT_ROOT / "correlation_results_paper.tex",
    caption="spearman correlation between translation quality and predicted MT labels across datasets and detectors",
    label="tab:quality-correlation",
)

In [ ]:
binned_rows = []

for metric in ["comet", "bleurt"]:
    sub_metric = merged[merged[metric].notna()].copy()

    for (language, dataset, detector), sub in sub_metric.groupby(["language", "dataset", "detector"]):
        if len(sub) < 10:
            continue
        try:
            sub = sub.copy()
            sub["quality_bin"] = make_quantile_bins(sub[metric], q=3)
        except Exception:
            continue

        agg = (
            sub.groupby("quality_bin", observed=False)
               .agg(
                   mean_quality=(metric, "mean"),
                   detection_rate=("pred_mt", "mean"),
                   n=("pred_mt", "size")
               )
               .reset_index()
        )
        agg["language"] = language
        agg["dataset"] = dataset
        agg["detector"] = detector
        agg["metric"] = metric.upper()
        binned_rows.append(agg)

binned_df = pd.concat(binned_rows, ignore_index=True)
binned_df.to_csv(OUTPUT_ROOT / "binned_quality_analysis.csv", index=False)

binned_paper = (
    binned_df.groupby(["metric", "detector", "quality_bin"], observed=False)
             .agg(
                 mean_quality=("mean_quality", "mean"),
                 detection_rate=("detection_rate", "mean"),
                 n=("n", "sum")
             )
             .reset_index()
)
binned_paper.to_csv(OUTPUT_ROOT / "binned_quality_analysis_paper.csv", index=False)

export_latex_table(
    binned_paper,
    OUTPUT_ROOT / "binned_quality_analysis_paper.tex",
    caption="average MT detection rate across low, medium, and high translation-quality bins",
    label="tab:quality-bins",
)

In [ ]:
translator_rows = []

for metric in ["comet", "bleurt"]:
    sub_metric = merged[merged[metric].notna()].copy()
    for (language, dataset, detector, translator), sub in sub_metric.groupby(["language", "dataset", "detector", "translator"]):
        translator_rows.append({
            "language": language,
            "dataset": dataset,
            "detector": detector,
            "translator": translator,
            "metric": metric.upper(),
            "avg_quality": sub[metric].mean(),
            "detection_rate": sub["pred_mt"].mean(),
            "n": len(sub),
        })

translator_df = pd.DataFrame(translator_rows)
translator_df.to_csv(OUTPUT_ROOT / "translator_quality_detection.csv", index=False)

translator_paper = translator_df[translator_df["metric"] == "COMET"].sort_values(
    ["language", "dataset", "detector", "avg_quality"]
)
translator_paper.to_csv(OUTPUT_ROOT / "translator_quality_detection_paper.csv", index=False)

export_latex_table(
    translator_paper,
    OUTPUT_ROOT / "translator_quality_detection_paper.tex",
    caption="average COMET score and MT detection rate by translator",
    label="tab:translator-quality",
)


In [ ]:
for metric in ["comet", "bleurt"]:
    plot_df = binned_df[binned_df["metric"] == metric.upper()].copy()
    if plot_df.empty:
        continue

    summary = (
        plot_df.groupby(["detector", "quality_bin"], observed=False)
               .agg(detection_rate=("detection_rate", "mean"))
               .reset_index()
    )

    bins = ["Low", "Medium", "High"]
    detectors = list(summary["detector"].unique())
    x = np.arange(len(bins))
    width = 0.8 / max(1, len(detectors))

    plt.figure(figsize=(12, 5))
    for i, detector in enumerate(detectors):
        sub = summary[summary["detector"] == detector].set_index("quality_bin").reindex(bins)
        plt.bar(
            x + i * width - (len(detectors)-1)*width/2,
            sub["detection_rate"],
            width=width,
            label=detector
        )

    plt.xticks(x, bins)
    plt.ylabel("MT detection rate")
    plt.xlabel(f"{metric.upper()} quality bin")
    plt.title(f"MT detection across {metric.upper()} quality bins")
    plt.legend(fontsize="small")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f"{metric}_quality_bins_bar.png", dpi=300)
    plt.close()

scatter_df = translator_df[translator_df["metric"] == "COMET"].copy()
if not scatter_df.empty:
    plt.figure(figsize=(10, 6))
    for detector, sub in scatter_df.groupby("detector"):
        plt.scatter(sub["avg_quality"], sub["detection_rate"], label=detector, alpha=0.7)

    plt.xlabel("Average COMET")
    plt.ylabel("MT detection rate")
    plt.title("Translator quality vs MT detectability")
    plt.legend(fontsize="small")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "translator_comet_vs_detection.png", dpi=300)
    plt.close()

summary_rows = []
for metric in ["comet", "bleurt"]:
    sub_metric = merged[merged[metric].notna()].copy()
    for detector, sub in sub_metric.groupby("detector"):
        pr, pp, sr, sp = safe_corr(sub[metric], sub["pred_mt"])
        summary_rows.append({
            "detector": detector,
            "metric": metric.upper(),
            "spearman_r": sr,
            "pearson_r": pr,
            "n": len(sub),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "overall_quality_summary.csv", index=False)

summary_paper = summary_df.pivot(index="detector", columns="metric", values="spearman_r").reset_index()
summary_paper.to_csv(OUTPUT_ROOT / "overall_quality_summary_paper.csv", index=False)

export_latex_table(
    summary_paper,
    OUTPUT_ROOT / "overall_quality_summary_paper.tex",
    caption="overall spearman correlation between translation quality and MT detectability by detector",
    label="tab:overall-quality-summary",
)

print(f"Done. Outputs saved to: {OUTPUT_ROOT}")